In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType


In [33]:
# Initialize Spark with Kafka package
spark = SparkSession.builder \
 .appName("FraudDetection") \
 .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
 .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [34]:
# 1. Load Static User Data (CSV)
users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True
)
users_df.show()

+------+-------------+-------------------+--------+--------------+
|userId|         name|              email|   phone|account_status|
+------+-------------+-------------------+--------+--------------+
|   101|  Alice Smith|  alice@example.com|555-0101|        active|
|   102|    Bob Jones|    bob@example.com|555-0102|        active|
|   103|Charlie Brown|charlie@example.com|555-0103|        active|
|   104| Diana Prince|  diana@example.com|555-0104|        active|
|   105|  Evan Wright|   evan@example.com|555-0105|  under_review|
+------+-------------+-------------------+--------+--------------+



In [35]:
!pwd
!find / -name "user_table.csv" 2>/dev/null

/workspace/fraud-detection
/workspace/fraud-detection/data/user_table.csv


In [36]:
# 2. Read Streaming Data from Kafka
tx_schema = StructType([
 StructField("tx_id", IntegerType(), True),
 StructField("userId", IntegerType(), True),
 StructField("amount", DoubleType(), True),
 StructField("timestamp", DoubleType(), True)
])
kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "fraud-detection") \
 .load()

In [37]:
# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [38]:
# 4. Enrich Stream with User Details
enriched_fraud = fraud_stream.join(users_df, "userId")

In [39]:
# 5. Format for output Kafka topic
output_stream = enriched_fraud \
.withColumn("value", to_json(struct("*")).cast("string")) \
.select("value")

In [ ]:
# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "kafka:9092") \
.option("topic", "fraud-notification") \
.option("checkpointLocation", "/workspace/checkpoints") \
.start()
query.awaitTermination()